# Prepare data
This notebook serve the purpose of preparing the data for the training of the model. It includes dataset formatting to the right format, splitting the dataset, cleaning it by removing the cases that we marked as too damaged, computing the common foreground mask, and statistics about the training data that will be used for the data augmentation and the training of the model.

The dataset should have already been downloaded and extracted in the data directory. If not, please follow the instructions in the README to do so.
The FIVES dataset should be located in EVAPORE/data/FIVES, and should have the following structure:
- FIVES/
    - test/
        - Ground truth/
        - Original/
    - train/
        - Ground truth/
        - Original/
    - clean_files_idx.txt
    - (optional) Quality Assessment.xlsx

Note: The clean_files_idx.txt is provided in the git repository, and contains the list of indices of the clean files in the dataset.

In [1]:
import os

fives_dataset_dir = os.path.abspath('../data/FIVES')
print(f"FIVES dataset directory: {fives_dataset_dir}")

FIVES dataset directory: /home/morand/afs/EVAPORE/data/FIVES


In [2]:
import json

dataset_infos = {
    "name": "FIVES",
    "ndim": 2
}

dataset_infos_filename = "dataset_infos.json"
dataset_infos_filepath = os.path.join(fives_dataset_dir, dataset_infos_filename)

with open(dataset_infos_filepath, "w") as f:
    json.dump(dataset_infos, f, indent=4)

## Format the dataset to the accepted format
Here we keep the original data, but you can delete it if you want to save storage

In [ ]:
from tqdm import tqdm
from shutil import copyfile
import json

train_img_dir = os.path.join(fives_dataset_dir, "train", "Original")
train_gt_dir = os.path.join(fives_dataset_dir, "train", "Ground truth")
train_filenames_list = [f for f in os.listdir(train_img_dir) if f.endswith(('.png'))]
print(f"Number of training images: {len(train_filenames_list)}")
print(f"First 5 training images: {train_filenames_list[:5]}")

test_img_dir = os.path.join(fives_dataset_dir, "test", "Original")
test_gt_dir = os.path.join(fives_dataset_dir, "test", "Ground truth")
test_filenames_list = [f for f in os.listdir(test_img_dir) if f.endswith(('.png'))]
print(f"Number of testing images: {len(test_filenames_list)}")
print(f"First 5 testing images: {test_filenames_list[:5]}")

dst_img_dir = os.path.join(fives_dataset_dir, "img")
dst_gt_dir = os.path.join(fives_dataset_dir, "gt")
os.makedirs(dst_img_dir, exist_ok=True)
os.makedirs(dst_gt_dir, exist_ok=True)

splits = {}
train_idx = []
for filename in tqdm(train_filenames_list):
    i = int(filename.split('_')[0])
    id = f"FIVES_{i:03d}"
    train_idx.append(id)

    new_filename = f"{id}.png"
    src_img_path = os.path.join(train_img_dir, filename)
    dst_img_path = os.path.join(dst_img_dir, new_filename)
    copyfile(src_img_path, dst_img_path)

    src_gt_path = os.path.join(train_gt_dir, filename)
    dst_gt_path = os.path.join(dst_gt_dir, new_filename)
    copyfile(src_gt_path, dst_gt_path)

test_idx = []
for filename in tqdm(test_filenames_list):
    i = int(filename.split('_')[0]) + len(train_filenames_list)
    id = f"FIVES_{i:03d}"
    test_idx.append(id)

    new_filename = f"{id}.png"
    src_img_path = os.path.join(test_img_dir, filename)
    dst_img_path = os.path.join(dst_img_dir, new_filename)
    copyfile(src_img_path, dst_img_path)

    src_gt_path = os.path.join(test_gt_dir, filename)
    dst_gt_path = os.path.join(dst_gt_dir, new_filename)
    copyfile(src_gt_path, dst_gt_path)

splits['train'] = train_idx
splits['test'] = test_idx
splits_filepath = os.path.join(fives_dataset_dir, "splits.json")
with open(splits_filepath, 'w') as f:
    json.dump(splits, f, indent=4)

# Data cleaning
We import the clean file indices that we manually identified and add them to the splits

In [ ]:
clean_files_idx_filepath = os.path.join(fives_dataset_dir, "clean_files_idx.txt")
if not os.path.exists(clean_files_idx_filepath):
    raise ValueError(f"File {clean_files_idx_filepath} does not exist.")
with open(clean_files_idx_filepath, 'r') as f:
    clean_files_idx = [line.strip() for line in f.readlines()]

print(f"Number of clean files: {len(clean_files_idx)}")
print(f"First 5 clean file indices: {clean_files_idx[:5]}")

splits['train_clean'] = clean_files_idx

with open(splits_filepath, 'w') as f:
    json.dump(splits, f, indent=4)

## Foreground mask computation
Compute the common foreground mask across all images in the train set, which will be used to refine the predictions of the model by masking out irrelevant background areas.
This will also be used in the metrics computation to focus only on the relevant areas of the images.

In [ ]:
import numpy as np
from PIL import Image
from skimage.measure import label

img_dir = os.path.join(fives_dataset_dir, 'train/Original/')
img_paths = [os.path.join(img_dir, fname) for fname in os.listdir(img_dir) if fname.endswith('.png')]

threshold = 4
debug_i = 20

non_zero_masks = []
for i, img_path in enumerate(img_paths):
    if i >= debug_i:
        break

    img_gray = Image.open(img_path).convert("L")
    img_gray = np.array(img_gray)
    non_zero_mask = img_gray > threshold

    cc, num_cc = label(non_zero_mask, return_num=True, connectivity=2)
    cc_sizes = [np.sum(cc == i) for i in range(1, num_cc + 1)]
    max_cc_index = np.argmax(cc_sizes) + 1
    non_zero_mask = cc == max_cc_index

    non_zero_masks.append(non_zero_mask)

common_mask = np.logical_and.reduce(non_zero_masks)

In [ ]:
import matplotlib.pyplot as plt
import torch

plt.imshow(common_mask, cmap='gray')
plt.title(f"Common Mask Between First {debug_i} Images")
plt.axis('off')
plt.show()

foreground_mask_dir = os.path.join(fives_dataset_dir, "foreground_masks")
os.makedirs(foreground_mask_dir, exist_ok=True)

torch.save(torch.from_numpy(common_mask), os.path.join(foreground_mask_dir, "FIVES.pt"))

# Test and visualize the dataset

We intialize the dataset object, that is used to load the data. We also initialize the datamodule, that is used to split the data into train, val and test sets, apply the transformations and augmentations, and create the dataloaders.

In [ ]:
from image_segmentation.data import ImageDataset, ImageDatamodule

dataset = ImageDataset(data_dir=fives_dataset_dir, transforms=None)
datamodule = ImageDatamodule(dataset, 
                             split_file_path=splits_filepath,
                             train_split_name='train_clean',
                             val_split_ratio=0.2,
                             train_transforms=None,
                             val_transforms=None,
                             test_transforms=None,
                             num_workers=0,
                             train_batch_size=4,
                             val_batch_size=1,
                             seed=42,
                             shuffle_train=True)
datamodule.setup()

We precompute the dataset statistics for the 'train_clean' split using the combined indices of train and val splits, without the test split, to ensure that the statistics reflect only the data used for training and validation, and not the unseen test data. This allows us to have a more accurate understanding of the data distribution that the model will be trained on, without any influence from the test set which is meant to evaluate generalization performance. 

We also compute thoses statistics on all the image and on only the foreground pixels, using the precomputed common foreground mask, to get a better insight into the distribution of pixel values in the regions of interest that the model will learn to segment.

In [ ]:
stats = dataset.get_dataset_stats(split_name='train_clean', split_indices=datamodule.train_indices.tolist() + datamodule.val_indices.tolist())
print("Dataset Statistics for 'train_clean' split:")
print(stats)

In [ ]:
dataloader = datamodule.train_dataloader()
img, gt = next(iter(dataloader))
print(f"Image batch shape: {img.shape}")
print(f"Ground truth batch shape: {gt.shape}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img[0].numpy())
plt.title("Sample Image")
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(gt[0].squeeze().numpy(), cmap='gray')
plt.title("Sample Ground Truth")
plt.axis('off')
plt.show()

After executing this notebook, the FIVES dataset folder should have the following structure:
- FIVES/
    - (optional) test/
        - Ground truth/
        - Original/
    - (optional) train/
        - Ground truth/
        - Original/
    - img/
    - gt/
    - foreground_masks/
    - image_stats.json
    - splits.json
    - clean_files_idx.txt
    - (optional) Quality Assessment.xlsx

Now that the dataset is in the right format, you can continue on the [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb)